### Copyright (C) Infineon Technologies AG 2025
 
Copyright (c) 2025, Infineon Technologies AG, or an affiliate of Infineon Technologies AG. All rights reserved.
This software, associated documentation and materials ("Software") is owned by Infineon Technologies AG or one of its affiliates ("Infineon") and is protected by and subject to worldwide patent protection, worldwide copyright laws, and international treaty provisions. Therefore, you may use this Software only as provided in the license agreement accompanying the software package from which you obtained this Software. If no license agreement applies, then any use, reproduction, modification, translation, or compilation of this Software is prohibited without the express written permission of Infineon.

Disclaimer: UNLESS OTHERWISE EXPRESSLY AGREED WITH INFINEON, THIS SOFTWARE IS PROVIDED AS-IS, WITH NO WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING, BUT NOT LIMITED TO, ALL WARRANTIES OF NON-INFRINGEMENT OF THIRD-PARTY RIGHTS AND IMPLIED WARRANTIES SUCH AS WARRANTIES OF FITNESS FOR A SPECIFIC USE/PURPOSE OR MERCHANTABILITY. Infineon reserves the right to make changes to the Software without notice. You are responsible for properly designing, programming, and testing the functionality and safety of your intended application of the Software, as well as complying with any legal requirements related to its use. Infineon does not guarantee that the Software will be free from intrusion, data theft or loss, or other breaches ("Security Breaches"), and Infineon shall have no liability arising out of any Security Breaches. Unless otherwise explicitly approved by Infineon, the Software may not be used in any application where a failure of the Product or any consequences of the use thereof can reasonably be expected to result in personal injury.

# In-Cabin Monitoring with MiniDMS Model

This notebook demonstrates a compact MiniDMS-style CNN for four in-cabin driver-behavior classes: Safe Driving, Talking Phone, Texting Phone, and Turning. It loads and splits the Multi-Class Driver Behavior Image Dataset, trains and evaluates the model, exports representative image tensors with the ONNX model, and converts it for embedded deployment on AURIX™ and TRAVEO™ targets.

## Notebook Structure

1. **Import libraries and helper functions**
2. **Load and explore the dataset**
3. **Visualize sample images**
4. **Create and build the MiniDMS model**
5. **Train the model**
6. **Evaluate model performance**
7. **Export the model and test data**
8. **Convert the model for deployment**
9. **Inference timing overview**
10. **Compare performance on different hardware platforms**
11. **References and Generated Artifacts**

## 1. Import libraries and helper functions

This section loads the required packages, dataset utilities, and any project-specific helper functions used for model definition, training, metrics, and export.

In [ ]:
import os
import sys
import matplotlib.pyplot as plt
from pathlib import Path

# Add the parent directory to the Python path so we can import from central_scripts
parent_dir = os.path.dirname(os.getcwd())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
from _CentralScripts.helper_functions import *
import modelling_helper as mh
from modelling_helper import CustomImageDataset

origin = "torch"

# Increase number of epochs to achieve better training results, recommended 20
EPOCHS = 1
LR = 1e-5
WD = 1e-2
LABEL_SMOOTHING = 0.1

DEVICE = "cpu"
CKPT_DIR = Path("checkpoints")
IMG_SIZE = 128
NUM_CLASSES = 4
WARMUP_EPOCHS = 100

DATASET_PATH = "./DriverMonitoringDataset"
TRAIN_PERCENTAGE = 0.6
VALID_PERCENTAGE = TRAIN_PERCENTAGE + 0.2

## 2. Load and explore the dataset

This section loads the local image dataset, validates readable image files, reports the class distribution, and creates reproducible train, validation, and test splits. The notebook excludes the `other_activities` class and uses the four driver-behavior classes Safe Driving, Talking Phone, Texting Phone, and Turning. Images are resized to 128 × 128 pixels; training uses horizontal flips and light color jitter, while validation and test data use resize and ImageNet normalization only.

**Source:** [Multi-Class Driver Behavior Image Dataset](https://data.mendeley.com/datasets/mzb4b6dff3.1/1), Sahin Afridi et al. (2024), Mendeley Data, version 1, DOI: [10.17632/mzb4b6dff3.1](https://doi.org/10.17632/mzb4b6dff3.1). The dataset is used for supervised image classification; verify the source dataset license and usage terms before redistribution or commercial deployment. The local `DriverMonitoringDataset` directory is the notebook's expected input artifact.

In [ ]:
from pathlib import Path
import random

from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms

dataset_root = mh.extract_local_in_cabin_zip(project_dir="./", extract_to="./")

DATASET_ROOT = Path(DATASET_PATH)
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ]
)

class_names = sorted(
    p.name
    for p in DATASET_ROOT.iterdir()
    if p.is_dir()
    and p.name.lower() != "other_activities"
    and p.name.lower() != ".ipynb_checkpoints"
)
print(f"class_names:   {class_names}")

if not class_names:
    raise FileNotFoundError(f"No class folders found under {DATASET_ROOT}")

class_to_idx = {name: idx for idx, name in enumerate(class_names)}


rng = random.Random(42)
train_samples, val_samples, test_samples = [], [], []

for class_name in class_names:
    class_dir = DATASET_ROOT / class_name
    image_files = sorted(
        p
        for p in class_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    )
    valid_files = []
    for img_path in image_files:
        try:
            with Image.open(img_path) as img:
                img.verify()
            valid_files.append(img_path)
        except Exception:
            continue

    if not valid_files:
        continue

    rng.shuffle(valid_files)
    train_end = int(len(valid_files) * TRAIN_PERCENTAGE)
    val_end = int(len(valid_files) * VALID_PERCENTAGE)

    for path in valid_files[:train_end]:
        train_samples.append((path, class_name))
    for path in valid_files[train_end:val_end]:
        val_samples.append((path, class_name))
    for path in valid_files[val_end:]:
        test_samples.append((path, class_name))

train_dataset = CustomImageDataset(
    train_samples, transform=train_transform, class_to_idx=class_to_idx
)
val_dataset = CustomImageDataset(
    val_samples, transform=val_transform, class_to_idx=class_to_idx
)
test_dataset = CustomImageDataset(
    test_samples, transform=val_transform, class_to_idx=class_to_idx
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {class_names}")

## 3. Visualize sample images

This section shows representative images from each retained class to confirm image readability, labeling, and visual coverage before training.

In [ ]:
# Use the local class directories directly for a quick visual sanity check.

fig, axes = plt.subplots(len(class_names), 2, figsize=(8, 10))

for row, class_name in enumerate(class_names):
    class_dir = DATASET_ROOT / class_name
    sample_files = sorted(class_dir.glob("*.*"))[:2]

    if not sample_files:
        axes[row, 0].axis("off")
        axes[row, 1].axis("off")
        axes[row, 0].text(
            0.5, 0.5, f"No images found\n{class_name}", ha="center", va="center"
        )
        continue

    for col, sample_file in enumerate(sample_files[:2]):
        img = plt.imread(sample_file)
        axes[row, col].imshow(img)
        axes[row, col].set_title(class_name.replace("_", " ").title())
        axes[row, col].axis("off")

# Hide unused axes if there are fewer than 2 columns in some rows
for ax in axes.flat:
    if not ax.has_data():
        ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Create and build the MiniDMS model

The MiniDMS architecture is a compact CNN with reduced depth, depthwise-separable convolutions, and a lightweight classification head. It is configured for the four retained classes and the 128 × 128 RGB input used by the preprocessing pipeline, making it suitable for embedded inference experiments.

In [ ]:
from modelling_helper import build_minidms
import torch
from torchsummary import summary

num_classes = len(class_names)
model = build_minidms(width_mult=0.5, num_classes=num_classes, verbose=True)
model = model.to(DEVICE)
summary(model, (3, IMG_SIZE, IMG_SIZE), device=DEVICE)

## 5. Train the model

Training defines the cross-entropy loss with label smoothing, AdamW optimizer, learning-rate schedule, checkpoint metadata, and validation loop. The configured epoch count is intentionally small for a quick example; increase it for a meaningful accuracy study and report the resulting training configuration with the evaluation results.

In [ ]:
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

lr = LR
wd = WD
patience = 15
model_name = "mini_half_dms"
phase2_name = "_phase2"

meta = {"model_name": model_name, "classes": class_names}

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
criterion = criterion.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

if WARMUP_EPOCHS > 0 and WARMUP_EPOCHS < EPOCHS:
    warmup = LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS
    )
    cosine = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)
    sched2 = SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS]
    )
else:
    sched2 = CosineAnnealingLR(optimizer, T_max=EPOCHS)

mh.train_phase(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    sched2,
    EPOCHS,
    torch.device("cpu"),
    phase2_name,
    extra_meta=meta,
    patience=patience,
    ckpt_dir=CKPT_DIR,
    img_size=IMG_SIZE,
    num_classes=NUM_CLASSES,
)

## 6. Evaluate model performance

The validation confusion matrix and accuracy provide the model-quality baseline. Review per-class errors and class balance before interpreting deployment results; inference timing measures execution cost and does not establish classification quality.

In [ ]:
accuracy, cm = mh.plot_confusion_matrix(
    model,
    val_loader,
    device=DEVICE,
    class_names=["Safe Driving", "Talking Phone", "Texting Phone", "Turning"],
    title="Validation Confusion Matrix",
)

## 7. Export the model and test data

Export the trained model to ONNX together with a representative validation image tensor and expected output label. These artifacts define the input/output contract used for conversion validation and embedded deployment.

In [ ]:
images, labels = next(iter(val_loader))
input_target = torch.squeeze(images[:1].to(DEVICE), 0)
output_target = torch.Tensor.numpy(labels[:1].to(DEVICE), force=True)

save_all(model_name, input_target, output_target, model, origin)
print("Model exported to ONNX")

## 8. Convert the model for deployment

Start the Docker conversion service, submit the exported ONNX model and representative test data, and generate target-specific source, binary, log, timing, and profile files. The supported deployment targets are AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4. Results are saved in `out/<model_name>/test_<model_name>/<target>/`.

In [ ]:
ensure_docker_container()

In [ ]:
from _CentralScripts.python_flask_client import CallTools

model_folder, onnx_model_file = get_output_paths(model_name)

target = "tc4dx_ppu"  # Available targets: "tc3xx", "tc4dx", "tc4dx_ppu", "arm_m4"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
    profile=True,
)
tool.convert_model()

## 9. Inference timing overview

Summarize the selected target's `results.json` timing section, including estimated cycles, instruction count, average CPI, latency, and throughput. Profile shares describe operator contributions to execution cost and are separate from classification accuracy.

In [ ]:
display_inference_benchmark(model_folder, target)

In [ ]:
plot_profile_shares(model_folder, target)

## 10. Compare performance on different hardware platforms

Run the comparison target and display platform-specific inference timing and resource results for the exported MiniDMS model. Use these results to compare deployment trade-offs across AURIX™ and TRAVEO™ targets.

In [ ]:
target = "compare"

tool = CallTools(
    folder=model_folder,
    url="http://localhost:8080/convert",
    target=target,
    tsim=False,
)
tool.convert_model()

In [ ]:
df = display_platform_comparison(model_folder)

## 11. References and Generated Artifacts

- **Dataset:** [Multi-Class Driver Behavior Image Dataset](https://data.mendeley.com/datasets/mzb4b6dff3.1/1), Sahin Afridi et al. (2024), Mendeley Data, version 1, DOI: [10.17632/mzb4b6dff3.1](https://doi.org/10.17632/mzb4b6dff3.1).
- **Model artifacts:** the `mini_half_dms` ONNX model, representative 128 × 128 RGB input tensor, expected output label, and checkpoint metadata.
- **Evaluation artifacts:** validation accuracy, confusion matrix, class names, and training checkpoints.
- **Deployment artifacts:** generated source, binaries, logs, timing/profile results, and platform comparison output under `out/mini_half_dms/test_mini_half_dms/<target>/`.
- **Deployment targets:** AURIX TC3x, AURIX TC4x, AURIX TC4x PPU, and TRAVEO Cortex M4.
- **Limitations:** the local split and short example training run are intended for demonstration. Validate dataset licensing, class coverage, lighting/camera variation, and safety requirements before deployment.